# 07 · Buffalo model validation

Independent checks of the buffalo habitat model. This notebook is **self-contained**: it downloads and builds the dataset itself with `buffalo_data.py`, the same code 06 uses, so it doesn't need 06 to have run.

1. **Coordinate system:** is EPSG:32736 (UTM 36S, ADR-0001) accurate enough over the whole study area?
2. **Spatially blocked cross-validation:** whole regions are held out, not random points. Reports AUC, TSS and calibration.
3. **Own-year embeddings:** each record carries the AlphaEarth embedding of its own year.
4. **Camera traps, region held out:** cameras scored by a model that never saw their region.

Not included: a benchmark against the Carrera-Faja et al. (2023) MaxEnt map, whose raster isn't published, and a census-density check, because TAWIRI/KWS census data aren't public.

## 0. Build the dataset

`build_buffalo_dataset` repeats 06's pipeline: GBIF buffalo and mammal sightings, thinning, visited cells, background, AlphaEarth embeddings and camera traps. It downloads everything fresh, which takes several minutes (about 16,000 GBIF records and 7,000 Earth Engine points). To reuse downloads between runs, use `BuffaloConfig(cache_dir="cache")`. Cached files are git-ignored parquet; delete the folder to refresh.

In [1]:
import ee
import numpy as np
import pandas as pd
from pyproj import CRS, Proj, Transformer

from buffalo_data import BAND_COLS, BuffaloConfig, build_buffalo_dataset, get_range_year_reduced_area_embeddings

EE_PROJECT = "project-0cf410a3-f35f-4911-9c5"
ee.Initialize(project=EE_PROJECT)

config = BuffaloConfig()
data = build_buffalo_dataset(config)

sdm_df = data.sdm_df
cam = data.cameras
band_cols = BAND_COLS
min_lon, min_lat, max_lon, max_lat = config.bounds

retry 1/5 at offset 0: 503 Server Error: Backend fetch failed for url: https://api.gbif.org/v1/occurrence/search?scientificName=Syncerus+caffer&country=TZ&hasCoordinate=true&hasGeospatialIssue=false&offset=0&limit=300
retry 2/5 at offset 0: 503 Server Error: Backend fetch failed for url: https://api.gbif.org/v1/occurrence/search?scientificName=Syncerus+caffer&country=TZ&hasCoordinate=true&hasGeospatialIssue=false&offset=0&limit=300
buffalo sightings in box: 674
presences after thinning: 658
2017 1353
2018 1693
2019 1800
2020 823
2021 662
2022 1435
2023 2210
2024 2765
2025 3132
mammal sightings in box: 15873
visited cells: 334
2017: 495 points embedded
2018: 792 points embedded
2019: 792 points embedded
2020: 396 points embedded
2021: 341 points embedded
2022: 737 points embedded
2023: 968 points embedded
2024: 1320 points embedded
2025: 1397 points embedded
points missing embeddings: 0
2017: 222 points embedded
dataset: 658 presences, 6580 background, 222 cameras


## 1. Is UTM 36S right for the whole study area?

Tanzania spans three UTM zones (35S, 36S, 37S), so a single UTM zone can't serve the whole country. What matters here is the **study area**: every layer must fall inside zone 36S, and the distortion must stay small everywhere in it.

- **Zone check:** each layer's extent is compared with the zone's official area of use (30–36°E, south of the equator).
- **Distortion:** UTM keeps shapes but stretches distances by a scale factor *k*. That's 0.9996 on the central meridian (33°E) and it grows toward the zone edges. `pyproj` gives *k* and the area factor at a 41 × 41 grid of points across the box. The errors are then shown at the scales this project uses: the 10 km blocks and the embedding buffer.

Earth Engine buffers are computed in meters on the Earth itself, not in UTM, so the embeddings don't depend on this choice.

In [2]:
UTM = "EPSG:32736"
zone = CRS(UTM).area_of_use

layers = {
    "study box": (np.array([min_lon, max_lon]), np.array([min_lat, max_lat])),
    "presences": (sdm_df.loc[sdm_df["presence"] == 1, "decimalLongitude"], sdm_df.loc[sdm_df["presence"] == 1, "decimalLatitude"]),
    "background": (sdm_df.loc[sdm_df["presence"] == 0, "decimalLongitude"], sdm_df.loc[sdm_df["presence"] == 0, "decimalLatitude"]),
    "camera traps": (cam["decimalLongitude"], cam["decimalLatitude"]),
}
zone_table = pd.DataFrame([
    {"layer": name,
     "lon": f"{np.min(lon):.2f} to {np.max(lon):.2f}",
     "lat": f"{np.min(lat):.2f} to {np.max(lat):.2f}",
     "inside zone 36S": bool(np.min(lon) >= zone.west and np.max(lon) <= zone.east and np.max(lat) <= zone.north)}
    for name, (lon, lat) in layers.items()
])
print(f"zone 36S area of use: {zone.west}..{zone.east} E, {zone.south}..{zone.north} N")
print(zone_table.to_string(index=False))

lon_g, lat_g = np.meshgrid(np.linspace(min_lon, max_lon, 41), np.linspace(min_lat, max_lat, 41))
factors = Proj(UTM).get_factors(lon_g.ravel(), lat_g.ravel())
k = np.asarray(factors.meridional_scale)
area_scale = np.asarray(factors.areal_scale)
max_dist_err = np.abs(k - 1).max()
max_area_err = np.abs(area_scale - 1).max()

print(f"scale factor k: {k.min():.5f} to {k.max():.5f}")
print(f"max distance error: {max_dist_err:.3%} -> {max_dist_err * config.cell_size_m:.0f} m on a {config.cell_size_m / 1000:.0f} km block, "
      f"{max_dist_err * config.buffer_m:.1f} m on the {config.buffer_m} m buffer")
print(f"max area error: {max_area_err:.3%}")

zone 36S area of use: 30.0..36.0 E, -80.0..0.0 N
       layer            lon            lat  inside zone 36S
   study box 34.00 to 36.00 -3.60 to -1.60             True
   presences 34.09 to 36.00 -3.60 to -1.60             True
  background 34.00 to 36.00 -3.60 to -1.60             True
camera traps 34.75 to 35.16 -2.64 to -2.39             True
scale factor k: 0.99975 to 1.00098
max distance error: 0.098% -> 10 m on a 10 km block, 0.7 m on the 700 m buffer
max area error: 0.196%


**Reading it:** if every layer is inside zone 36S and the maximum distance error is a small fraction of a percent, UTM 36S is accurate enough for every distance and area in this study, and nothing needs reprojecting. That condition only breaks if the study area grows east of 36°E (e.g. Tarangire, Kilimanjaro) or west of 30°E. That would need a different CRS and an update to ADR-0001.

## 2. Spatially blocked cross-validation

Sightings are spatially clustered, so a random split puts near-twins on both sides and overstates performance. Two blocking schemes, both scoring the **same XGBoost settings as 06** on the 64 embedding bands only:

- **10 km blocks** (as in 06): many small blocks, 5 folds.
- **4 quadrants:** the box is cut west/east and north/south at the **median presence position**, so each region holds about a quarter of the presences. Each quadrant is held out once, and the model has to predict a whole region it never saw.

For every held-out fold:
- **AUC:** does the model rank presences above background?
- **TSS** (true skill statistic = sensitivity + specificity − 1, from −1 to 1, with 0 = no skill): needs a cut-off between "suitable" and "not". The cut-off is chosen **on the training regions only**, by maximizing TSS on out-of-fold predictions from an inner 10 km block CV, then applied unchanged to the held-out region. Picking it on the test region would make TSS optimistic.
- **Calibration:** the Brier score, a calibration slope, and a table of predicted vs observed presence share.

In [3]:
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold

to_utm = Transformer.from_crs("EPSG:4326", UTM, always_xy=True)
sdm_df["x"], sdm_df["y"] = to_utm.transform(sdm_df["decimalLongitude"].values, sdm_df["decimalLatitude"].values)

X = sdm_df[band_cols]
y = sdm_df["presence"]

size = config.cell_size_m
sdm_df["cell"] = (sdm_df["x"] // size).astype(int).astype(str) + "_" + (sdm_df["y"] // size).astype(int).astype(str)

mid_x = sdm_df.loc[y == 1, "x"].median()
mid_y = sdm_df.loc[y == 1, "y"].median()
north_south = np.where(sdm_df["y"] >= mid_y, "north", "south")
east_west = np.where(sdm_df["x"] >= mid_x, "east", "west")
sdm_df["quadrant"] = [f"{ns}-{ew}" for ns, ew in zip(north_south, east_west)]

mid_lon, mid_lat = Transformer.from_crs(UTM, "EPSG:4326", always_xy=True).transform(mid_x, mid_y)
print(f"quadrant split at lon {mid_lon:.3f}, lat {mid_lat:.3f}")
print(pd.crosstab(sdm_df["quadrant"], sdm_df["presence"]).rename(columns={0: "background", 1: "presence"}))

quadrant split at lon 35.451, lat -3.061
presence    background  presence
quadrant                        
north-east         889        54
north-west        4312       275
south-east         723       275
south-west         656        54


In [4]:
from buffalo_validation import blocked_cv, make_model

block_splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, y, groups=sdm_df["cell"]))
block_table, block_scores, _ = blocked_cv(sdm_df, block_splits, [f"fold {i}" for i in range(5)])

quadrant_splits = list(LeaveOneGroupOut().split(X, y, groups=sdm_df["quadrant"]))
quadrant_names = [sdm_df["quadrant"].iloc[test[0]] for _, test in quadrant_splits]
quadrant_table, quadrant_scores, quadrant_models = blocked_cv(sdm_df, quadrant_splits, quadrant_names)

print("10 km blocks:")
print(block_table)
print()
print("4 quadrants (each region held out once):")
print(quadrant_table)

C:\Users\Juan\AppData\Local\Temp\ipykernel_14468\1977898892.py:37: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype int64 cannot hold the value being set, so the result has dtype float64. Cast the object to float64 before this operation to retain the current behavior.
  table.loc["mean"] = table.mean(numeric_only=True)


10 km blocks:
          presences  background    AUC    TSS  cut-off  Brier
held out                                                     
fold 0        134.0        1320  0.686  0.217    0.005  0.090
fold 1        139.0        1307  0.744  0.368    0.006  0.091
fold 2        137.0        1321  0.785  0.349    0.004  0.086
fold 3        126.0        1318  0.745  0.327    0.010  0.083
fold 4        122.0        1314  0.719  0.297    0.010  0.088
mean          131.6        1316  0.736  0.312    0.007  0.087

4 quadrants (each region held out once):
            presences  background    AUC    TSS  cut-off  Brier
held out                                                       
north-east       54.0         889  0.662  0.275    0.013  0.092
north-west      275.0        4312  0.590  0.109    0.020  0.067
south-east      275.0         723  0.522  0.061    0.005  0.265
south-west       54.0         656  0.684  0.250    0.016  0.075
mean            164.5        1645  0.615  0.174    0.014  0.125


C:\Users\Juan\AppData\Local\Temp\ipykernel_14468\1977898892.py:37: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype int64 cannot hold the value being set, so the result has dtype float64. Cast the object to float64 before this operation to retain the current behavior.
  table.loc["mean"] = table.mean(numeric_only=True)


**Reading it:**
- **AUC/TSS lower for quadrants than for 10 km blocks:** the model transfers less well to a whole unseen region than to a gap inside known regions. The quadrant numbers are the honest estimate for mapping areas without sightings.
- **One quadrant much worse than the others:** buffalo habitat there looks different in the embeddings, or that quadrant's sightings are dominated by a different kind of place (e.g. lodges vs open plains).
- **TSS:** above about 0.4 is usually read as useful, and 0 means no skill.

### Calibration

The scores are **not** meant to be probabilities of buffalo presence, so the predicted and observed columns aren't expected to agree. The "observed" share is the fraction of presences in the sample, which is fixed at about 1 in 11 by the 10:1 background design. `scale_pos_weight` re-weights the classes on top of that. So the **level** of the scores (the intercept) means nothing on its own. What matters is the **calibration slope**: a logistic regression of presence on the logit of the score, using held-out scores only.
- **Slope ≈ 1:** score differences carry the right amount of information.
- **Slope < 1:** the model is over-confident, and its scores are too spread out.
- **Slope > 1:** it's under-confident.

For real probabilities, see the camera-trap calibration in 06.

In [5]:
from sklearn.linear_model import LogisticRegression


def calibration_report(score, name):
    logit = np.log(np.clip(score, 1e-4, 1 - 1e-4) / (1 - np.clip(score, 1e-4, 1 - 1e-4))).values.reshape(-1, 1)
    slope = LogisticRegression(C=np.inf).fit(logit, y).coef_[0, 0]
    bins = pd.qcut(score, 10, duplicates="drop")
    table = pd.DataFrame({"predicted": score, "observed": y}).groupby(bins, observed=True).mean()
    print(f"{name}: Brier {brier_score_loss(y, score):.3f}, calibration slope {slope:.2f}")
    print((table * 100).round(1).astype(str) + " %")
    print()


calibration_report(block_scores, "10 km blocks")
calibration_report(quadrant_scores, "4 quadrants")

10 km blocks: Brier 0.087, calibration slope 0.34
                        predicted observed
(-0.00099736, 0.000332]     0.0 %    2.1 %
(0.000332, 0.000942]        0.1 %    2.5 %
(0.000942, 0.00209]         0.1 %    3.6 %
(0.00209, 0.00414]          0.3 %    4.8 %
(0.00414, 0.00779]          0.6 %    5.1 %
(0.00779, 0.0151]           1.1 %    5.9 %
(0.0151, 0.0301]            2.1 %    9.3 %
(0.0301, 0.0675]            4.7 %   14.8 %
(0.0675, 0.201]            12.0 %   16.3 %
(0.201, 0.99]              45.9 %   26.5 %

4 quadrants: Brier 0.098, calibration slope 0.07
                        predicted observed
(-0.00099912, 0.000153]     0.0 %    7.2 %
(0.000153, 0.000474]        0.0 %    7.7 %
(0.000474, 0.00116]         0.1 %    9.4 %
(0.00116, 0.00247]          0.2 %    8.4 %
(0.00247, 0.00496]          0.4 %    8.1 %
(0.00496, 0.0101]           0.7 %    8.6 %
(0.0101, 0.0207]            1.5 %    8.9 %
(0.0207, 0.0494]            3.3 %    7.7 %
(0.0494, 0.145]             8.6 %   11.2

## 3. Does every record carry the embedding of its own year?

AlphaEarth has one embedding image per year, and 06 extracts each point from the image of the point's `year`. A mix-up there (an off-by-one year, or the wrong image for a group) would silently mix years. Check:

1. Draw a random sample of presences and background points (`YEAR_CHECK_N`).
2. Re-extract each from Earth Engine with the **same function** the dataset build uses, once from its own year and once from a neighbouring year.
3. The stored values must match the own-year extraction (difference ≈ 0) and **differ** from the neighbouring year. If they also matched the neighbouring year, the check couldn't tell years apart.

This needs Earth Engine: about 2 small requests per year in the sample.

In [6]:
YEAR_CHECK_N = 40
MATCH_TOLERANCE = 1e-6


def extract(points, year):
    """Same recipe as the dataset build, forced to one year."""
    info = get_range_year_reduced_area_embeddings(points.assign(year=year), "point_id", year, year, config=config)
    return pd.DataFrame([f["properties"] for f in info[0]["features"]]).set_index("point_id")[band_cols]


sample = pd.concat([sdm_df[sdm_df["presence"] == k].sample(YEAR_CHECK_N // 2, random_state=42) for k in (0, 1)])

rows = []
for year, group in sample.groupby("year"):
    year = int(year)
    other_year = year - 1 if year > config.start_year else year + 1
    stored = group.set_index("point_id")[band_cols]
    own = extract(group, year).loc[stored.index]
    other = extract(group, other_year).loc[stored.index]
    for point_id in stored.index:
        rows.append({"point_id": point_id, "year": year, "other year": other_year,
                     "diff own year": (stored.loc[point_id] - own.loc[point_id]).abs().max(),
                     "diff other year": (stored.loc[point_id] - other.loc[point_id]).abs().max()})

year_check = pd.DataFrame(rows)
year_check["matches own year"] = year_check["diff own year"] < MATCH_TOLERANCE
year_check["matches other year"] = year_check["diff other year"] < MATCH_TOLERANCE

print(f"points checked: {len(year_check)} across {year_check['year'].nunique()} years")
print("match their own year:", year_check["matches own year"].sum())
print("match the neighbouring year:", year_check["matches other year"].sum(), "(should be 0)")
print(f"smallest difference to the neighbouring year: {year_check['diff other year'].min():.4f}")
year_check.sort_values("diff own year", ascending=False).head()

2017: 2 points embedded
2018: 2 points embedded
2018: 2 points embedded
2017: 2 points embedded
2019: 6 points embedded
2018: 6 points embedded
2022: 4 points embedded
2021: 4 points embedded
2023: 11 points embedded
2022: 11 points embedded
2024: 7 points embedded
2023: 7 points embedded
2025: 8 points embedded
2024: 8 points embedded
points checked: 40 across 7 years
match their own year: 40
match the neighbouring year: 0 (should be 0)
smallest difference to the neighbouring year: 0.0491


,point_id,year,other year,diff own year,diff other year,matches own year,matches other year
23,4506350525,2023,2022,3.505529e-14,0.190166,True,False
34,bg_1225,2025,2024,2.739475e-14,0.149923,True,False
15,bg_2484,2023,2022,2.037259e-14,0.208042,True,False
39,5230365942,2025,2024,2.037259e-14,0.101045,True,False
35,bg_393,2025,2024,2.031708e-14,0.115309,True,False


## 4. Camera traps scored by a model that never saw their region

The camera check in 06 used the final model, trained on **all** sightings, including GBIF sightings right next to the cameras. Here each camera is scored by the quadrant model whose **held-out quadrant contains that camera**. The model never saw a single presence or background point from the camera's region, which makes this an out-of-region test against independent, effort-corrected data.

- The same comparison with a model trained on all data is shown next to it. If the held-out-region correlation is much lower, part of 06's camera result came from nearby GBIF sightings, not from transferable habitat signal.
- **Detection rate** = buffalo captures per 100 trap-days, as in 06. Camera embeddings are from 2017, the closest year to the 2010–2013 camera period.

In [7]:
from scipy.stats import spearmanr

cam["x"], cam["y"] = to_utm.transform(cam["decimalLongitude"].values, cam["decimalLatitude"].values)
cam["quadrant"] = [f"{'north' if cy >= mid_y else 'south'}-{'east' if cx >= mid_x else 'west'}" for cx, cy in zip(cam["x"], cam["y"])]
print("cameras per quadrant:")
print(cam["quadrant"].value_counts())

cam["held_out_score"] = np.nan
for quadrant, group in cam.groupby("quadrant"):
    cam.loc[group.index, "held_out_score"] = quadrant_models[quadrant].predict_proba(group[band_cols])[:, 1]

cam["all_data_score"] = make_model(y).fit(X, y).predict_proba(cam[band_cols])[:, 1]

summary = []
for column, label in [("held_out_score", "region held out"), ("all_data_score", "trained on all data")]:
    rho, p_value = spearmanr(cam[column], cam["rate"])
    groups = pd.qcut(cam[column], 3, labels=["low", "medium", "high"])
    medians = cam.groupby(groups, observed=True)["rate"].median()
    summary.append({"model": label, "Spearman rho": round(rho, 2), "p": round(p_value, 4),
                    "median rate low": round(medians["low"], 2), "median rate medium": round(medians["medium"], 2),
                    "median rate high": round(medians["high"], 2)})
pd.DataFrame(summary).set_index("model")

cameras per quadrant:
quadrant
north-west    222
Name: count, dtype: int64


,Spearman rho,p,median rate low,median rate medium,median rate high
model,,,,,
region held out,-0.01,0.8999,3.17,6.15,4.37
trained on all data,0.35,0.0000,1.53,3.67,9.93


**Reading it:**
- **Positive rho with the region held out**, and median detection rates rising from low to high: the habitat signal **transfers** to a region the model never saw. That's the strongest evidence in this notebook that the model learned buffalo habitat, not where tourists drive.
- **Rho near 0 with the region held out but positive with all data:** 06's camera result leaned on nearby sightings, and the model doesn't transfer well across regions.
- As in 06, the p-value is optimistic, because neighbouring cameras (about 2 km apart) aren't independent.